# PoetryDuel-GPT v4.1 — Two-Loss + Genre Token

**749K window-1 pairs (Lục Bát + Thất Ngôn), weighted TN loss 2.6x, linebreak suppression.**

| Step | Time |
|---|---|
| Download data + train tokenizer | ~2 min |
| Training (10K steps) | ~2.5 hr T4 |
| Generate poetry | ~1 min |

### v4.1 Changes:
- **Format:** `<|start|> [LUC_BAT]/[THAT_NGON] [RHYME:X] [TONE:...]` (no `[DOI_THO]`)
- **T2a:** Inference suppresses premature `<|linebreak|>` for Thất Ngôn until 7+ syllables
- **T2b:** TN training examples weighted 2.6x in loss (compensates 2.6:1 data ratio)
- **P1:** Rhyme constraint at pos6 (LB) / pos7 (TN)
- **P2:** Overgeneration fix

### v4.1 Targets
| Metric | Target |
|---|---|
| LB Syllable (6+8) | 90%+ |
| TN Syllable (7+7) | 80%+ |
| Rhyme (combined) | 85%+ |
| Stress test | 100% |

In [ ]:
# @title 1. Clone Repo + Install
!git clone https://github.com/weseegod/poetry-dual-gpt.git /content/poetry-dual-gpt 2>/dev/null
%cd /content/poetry-dual-gpt
!git pull origin main

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q pandas tokenizers tqdm datasets
!mkdir -p checkpoints

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print('\nRun cells in order.')


In [ ]:
# @title 2. Download Data + Train Tokenizer (~2 min)
from google.colab import drive
drive.mount('/content/drive')

# Copy data.zip from Drive
!cp /content/drive/MyDrive/poetry-dual-gpt/data.zip /content/poetry-dual-gpt/ 2>/dev/null
!unzip -o data.zip

!mkdir -p data
!mv doi_tho_corpus.txt data/doi_tho_corpus.txt 2>/dev/null
!ls data/

# Train BPE tokenizer on v4 corpus
%cd /content/poetry-dual-gpt
!python src/train_bpe.py --corpus data/doi_tho_corpus.txt

# Verify
from tokenizers import Tokenizer
tok = Tokenizer.from_file('tokenizer/poetry_bpe.model')
print(f'Vocab: {tok.get_vocab_size():,}')
for name in ['<|pad|>', '<|start|>', '<|reply|>', '<|end|>', '[LUC_BAT]', '[THAT_NGON]', '<|linebreak|>']:
    n = len(tok.encode(name).ids)
    print(f'  {name:20s} subwords={n} {"OK" if n==1 else "FAIL"}')

# Verify both genres present (no [DOI_THO] in v4.1 format)
import re
with open('data/doi_tho_corpus.txt', encoding='utf-8') as f:
    corpus = f.read()
lb_count = len(re.findall(r'\[LUC_BAT\]', corpus))
tn_count = len(re.findall(r'\[THAT_NGON\]', corpus))
print(f'\nLục Bát pairs: {lb_count:,}')
print(f'Thất Ngôn pairs: {tn_count:,}')

print("\nReady for training.")


In [ ]:
# @title 3. Train — Weighted TN Loss, 10K steps (~2.5 hr T4)
assert torch.cuda.is_available(), "Enable GPU: Runtime -> Change runtime type -> T4 GPU"

%cd /content/poetry-dual-gpt
import glob

# v4.1: 749K window-1 pairs, TN weighted 2.6x in loss
CORPUS = 'data/doi_tho_corpus.txt'

latest = sorted(glob.glob("checkpoints/doi_tho_step_*.pt"))
if latest:
    print(f"Resuming from {latest[-1]}")
    !python src/train.py --mode train --name doi_tho_ --corpus {CORPUS} --resume {latest[-1]}
else:
    !python src/train.py --mode train --name doi_tho_ --corpus {CORPUS}

print("\nTraining complete -> checkpoints/doi_tho_best.pt")


In [ ]:
# @title Verify: Test Both Genres
%cd /content/poetry-dual-gpt
import os, glob

ckpt = 'checkpoints/doi_tho_best.pt'
if not os.path.exists(ckpt):
    candidates = sorted(glob.glob('checkpoints/doi_tho_*.pt'), reverse=True)
    for c in candidates:
        if os.path.getsize(c) > 1000000:
            ckpt = c; break

if not os.path.exists(ckpt):
    print('No checkpoint found.')
else:
    print(f'Testing: {ckpt}\n')
    
    print('--- Luc Bat (6+8) ---')
    !python src/sample.py --checkpoint {ckpt} \
        --prompt "Than em nhu chen lua dong | Phat pho duoi ngon nang hong ban mai" \
        --num_samples 2 --device cuda
    
    print('\n--- That Ngon (7+7) ---')
    !python src/sample.py --checkpoint {ckpt} \
        --prompt "Ao thu lanh leo nuoc trong veo | Mot chiec thuyen cau be teo teo" \
        --num_samples 2 --device cuda
    
    print('\nExpected: LB 6+8, TN 7+7. Check syllable counts.')


In [ ]:
# @title Generate Custom
%cd /content/poetry-dual-gpt

# Luc Bat (use | between lines)
!python src/sample.py \
    --checkpoint checkpoints/doi_tho_best.pt \
    --prompt "Than em nhu chen lua dong | Phat pho duoi ngon nang hong ban mai" \
    --temperature 0.75 --top_p 0.92 --num_samples 2 --device cuda

# That Ngon
!python src/sample.py \
    --checkpoint checkpoints/doi_tho_best.pt \
    --prompt "Ao thu lanh leo nuoc trong veo | Mot chiec thuyen cau be teo teo" \
    --temperature 0.75 --top_p 0.92 --num_samples 2 --device cuda


In [ ]:
# @title Save to Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/poetry-dual-gpt'
!mkdir -p {DRIVE_DIR}/checkpoints

!cp -v tokenizer/poetry_bpe.model {DRIVE_DIR}/
import glob
for ckpt in sorted(glob.glob('checkpoints/doi_tho_*.pt')):
    !cp -v {ckpt} {DRIVE_DIR}/checkpoints/
print('Saved to Drive.')
